## Load the model and metadata

In [1]:
import joblib
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb


from config import MODEL_DIR, PROCESSED_DIR

In [2]:
saved = joblib.load(
    MODEL_DIR / "electricity_lightgbm_full.joblib"
)

model = saved["model"]
features = saved["features"]

print("Model:", saved["model_name"])
print("Features:", features)
print("Training period:", saved["training_period"])
print("Validation period:", saved["validation_period"])
print("Test period:", saved["test_period"])

Model: LightGBM with Weather
Features: ['building_id', 'site_id', 'primary_use', 'square_feet', 'hour', 'day_of_week', 'month', 'air_temperature', 'dew_temperature', 'wind_speed']
Training period: 2016-01 to 2016-09
Validation period: 2016-10 to 2016-11
Test period: 2016-12


## Load five random observations

In [3]:
parquet_path = (
    PROCESSED_DIR / "electricity_data.parquet"
).as_posix()

In [4]:
con = duckdb.connect()

con.execute("SELECT setseed(0.42)")

sample = con.execute(
    """
    SELECT *
    FROM read_parquet(?)
    WHERE timestamp >= '2016-12-01'
      AND timestamp < '2017-01-01'
    ORDER BY RANDOM()
    LIMIT 5
    """,
    [parquet_path]
).df()

display(sample)

,building_id,site_id,primary_use,square_feet,timestamp,hour,day_of_week,month,air_temperature,dew_temperature,wind_speed,meter_reading
0,975,9,Entertainment/public assembly,426472,2016-12-26 13:00:00,13,1,12,21.7,20.6,0.0,257.00
1,965,9,Lodging/residential,45086,2016-12-26 17:00:00,17,1,12,25.6,21.1,3.1,20.00
2,431,3,Parking,1760,2016-12-05 14:00:00,14,1,12,6.7,5.0,2.6,12.95
3,510,3,Public services,4581,2016-12-22 23:00:00,23,4,12,7.2,-3.9,5.7,4.28
4,1324,14,Entertainment/public assembly,84688,2016-12-07 13:00:00,13,3,12,4.4,2.8,3.1,13.00


## Prepare the model inputs

In [5]:
X_new = sample[features].copy()

categorical_features = [
    "building_id",
    "site_id",
    "primary_use"
]

for col in categorical_features:
    X_new[col] = X_new[col].astype("category")

numeric_features = [
    col for col in features
    if col not in categorical_features
]

for col in numeric_features:
    X_new[col] = X_new[col].astype("float32")

## Predict electricity consumption

In [6]:
pred_log = model.predict(X_new)

predicted = np.maximum(
    np.expm1(pred_log),
    0
)

comparison = pd.DataFrame({
    "Building ID": sample["building_id"].to_numpy(),
    "Actual": sample["meter_reading"].to_numpy(),
    "Predicted": predicted
})

display(comparison.round(2))

,Building ID,Actual,Predicted
0,975,257.00,456.67
1,965,20.00,28.88
2,431,12.95,17.27
3,510,4.28,8.44
4,1324,13.00,35.94


## Close DuckDB

In [7]:
con.close()